In [1]:
%pip install -q faiss-gpu

In [2]:
import os

os.environ["HF_TOKEN"] = input()

In [3]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")

dataset["train"]

Dataset({
    features: ['query', 'document_title', 'document_text', 'document_link', 'query_type', 'difficulty'],
    num_rows: 4612
})

In [4]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanum-cross-platform-retriever")

train_split = dataset["train"]

corpus_texts = list(dict.fromkeys(train_split["document_text"]))
corpus_dataset = Dataset.from_dict({"text": corpus_texts})

corpus_embeddings = retriever.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers.util import mine_hard_negatives
from datasets import Dataset

train_dataset = dataset["train"].select_columns(["query", "document_text"])
train_dataset = train_dataset.rename_columns({
    "query": "anchor",
    "document_text": "positive"
})

hard_train_dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,
    margin=0.02,
    range_min=0,
    range_max=100,
    sampling_strategy="top",
    batch_size=128,
    output_format="labeled-pair",
    use_faiss=True,
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Found 4603 unique queries out of 4612 total queries.
Found an average of 1.002 positives per query.


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00, 10.52it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           4,612         18,188               
Mean           0.5250         0.3872         0.1432
Median         0.5395         0.3790         0.1238
Std            0.1380         0.0973         0.1010
Min           -0.0667         0.1108         0.0200
25%            0.4351         0.3150         0.0548
50%            0.5395         0.3790         0.1238
75%            0.6261         0.4524         0.2109
Max            0.8639         0.7245         0.5469
Skipped 23,543 potential negatives (4.97%) due to the absolute_margin of 0.02.
Could not find enough negatives for 260 samples (1.13%). Consider adjusting the range_max and absolute_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4612
Enriched dataset size: 22764
{'anchor': 'compound bow explosives requirement', 'positive': 'Pyrotechnic Bow. Pyrotechnic Bow. The Pyrotechnic Bow Schematic can 

In [6]:
unique_queries = list(set(hard_train_dataset["anchor"]))
print(f"Total unique queries: {len(unique_queries)}")

Total unique queries: 4603


In [7]:
from sklearn.model_selection import train_test_split

train_queries, val_queries = train_test_split(
    unique_queries, test_size=0.1, random_state=42
)

In [8]:
train_set, val_set = set(train_queries), set(val_queries)
train_dataset = hard_train_dataset.filter(lambda x: x["anchor"] in train_set)
val_dataset   = hard_train_dataset.filter(lambda x: x["anchor"] in val_set)

print(f"Train rows: {len(train_dataset)}, queries: {len(train_set)}")
print(f"Val rows:   {len(val_dataset)},   queries: {len(val_set)}")

Filter:   0%|          | 0/22764 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22764 [00:00<?, ? examples/s]

Train rows: 20475, queries: 4142
Val rows:   2289,   queries: 461


In [9]:
train_dataset[0], train_dataset[161]

({'anchor': 'compound bow explosives requirement',
  'positive': 'Pyrotechnic Bow. Pyrotechnic Bow. The Pyrotechnic Bow Schematic can be found on the second level of the Black Mountain Clan. Expertise. Requires an expertise of 45 in Explosives Requires an expertise of 20 in Mechanical. Ingredients. Fire Obstruction + Compound Bow Fire Obstruction is the sixth degree in Explosvies The Compound Bow schematic can be purchased from Blacksmiths. Effects. One unit can be obtained per unit of both components Stats: 1-10 HD, 2-5 FT, 5-20 FD, 8 SP, 20 RNG, 5 WE, 80 QU. Used as an ingredient for:. None. Notes. The Mistress of Fire in Tulla carries a Pyrotechnic Bow. Item Page',
  'label': 1},
 {'anchor': 'healin jacket bug',
  'positive': "Heal. Heal. The Skill of Heal gives The Player the ability to cure wounds and remove crippling injury. Heal level is dependent upon Intelligence (IN). Training. Heal Apprentice: A Character may heal an extra 50% Trainer: Herbalists Trainer: Jongle Dunne in Shr

In [10]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    val_dataset["anchor"], val_dataset["positive"], val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(eval_samples)}")

Evaluation queries: 457


In [ ]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=eval_samples,
    at_k=10,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [18]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [19]:
results = evaluator(model)
print(results)

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

{'val-hard-negatives_map': 0.9104303428154632, 'val-hard-negatives_mrr@10': 0.9104303428154632, 'val-hard-negatives_ndcg@10': 0.9331647882317604}


In [14]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in train_dataset],
    "document": [p["positive"] for p in train_dataset],
    "label": [p["label"] for p in train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in val_dataset],
    "document": [p["positive"] for p in val_dataset],
    "label": [p["label"] for p in val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps = 4,

    logging_steps=25,

    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@10",
    greater_is_better=True,

    bf16=True, # check

    warmup_steps=0.1,
    lr_scheduler_type="linear",
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,

    evaluator=evaluator,
    loss=train_loss,
    # callbacks=[
    #         EarlyStoppingCallback(
    #                 early_stopping_patience=3,
    #                 early_stopping_threshold=0.001,
    #             )
    #     ],
)

In [16]:
trainer.train()

Step,Training Loss,Validation Loss,Val-hard-negatives Map,Val-hard-negatives Mrr@10,Val-hard-negatives Ndcg@10
25,0.349400,0.261151,0.931984,0.931984,0.949260
50,0.315101,0.253042,0.928045,0.929139,0.946902
75,0.268827,0.236183,0.937856,0.937856,0.953687
100,0.260761,0.228671,0.937053,0.937053,0.953127
125,0.267014,0.226188,0.941794,0.942159,0.956856
150,0.263794,0.222773,0.943910,0.943910,0.958272
175,0.264839,0.223772,0.947338,0.947702,0.960916
200,0.285522,0.233782,0.948104,0.948104,0.961402
225,0.282788,0.227746,0.941721,0.941721,0.956657
250,0.258737,0.213166,0.940263,0.940263,0.955563


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.2816784934997559, metrics={'train_runtime': 1658.024, 'train_samples_per_second': 12.349, 'train_steps_per_second': 0.193, 'total_flos': 0.0, 'train_loss': 0.2816784934997559, 'epoch': 0.78125})

In [17]:
model.push_to_hub("pameydorke/arcanumsearch-reranker-bge-reranker-v2-m3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/pameydorke/arcanumsearch-reranker-bge-reranker-v2-m3/commit/3776d78d7e1cd1d6b53ca822bb2a2490234fba87'

In [20]:
base_model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

ft_model = CrossEncoder("pameydorke/arcanumsearch-reranker-bge-reranker-v2-m3", num_labels=1)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/48.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

In [21]:
import math
import numpy as np

def get_per_query_ndcg(model, samples, k=10):
    """
    Computes NDCG@10 for each query in a list of samples.

    Args:
        model: A loaded CrossEncoder model.
        samples (list): List of dicts with 'query', 'positive', and 'negative' keys.
        k (int): The cutoff for NDCG.

    Returns:
        list: A list of NDCG@10 scores, one per query.
    """
    per_query_scores = []
    
    for sample in samples:
        query = sample["query"]
        positives = set(sample["positive"])
        # Combine all candidates (positives + negatives) for reranking
        candidates = list(positives) + sample.get("negative", [])
        
        # Get the cross-encoder's scores for all (query, candidate) pairs
        pairs = [(query, doc) for doc in candidates]
        scores = model.predict(pairs)
        
        # Sort candidates by score in descending order
        ranked_candidates = [doc for _, doc in sorted(zip(scores, candidates), reverse=True)]
        
        # Calculate DCG and IDCG for the top-k ranked list
        dcg = 0.0
        idcg = 0.0
        for i, doc in enumerate(ranked_candidates[:k]):
            rel = 1.0 if doc in positives else 0.0
            dcg += rel / math.log2(i + 2) # i+2 because rank starts at 1
            
        # IDCG is the score of a perfect ranking
        for i in range(min(len(positives), k)):
            idcg += 1.0 / math.log2(i + 2)
            
        ndcg = dcg / idcg if idcg > 0 else 0.0
        per_query_scores.append(ndcg)
        
    return per_query_scores

In [22]:
base_scores = np.array(get_per_query_ndcg(base_model, eval_samples))
ft_scores = np.array(get_per_query_ndcg(ft_model, eval_samples))

# Now you can proceed with the paired bootstrap
diffs = ft_scores - base_scores
rng = np.random.default_rng(42)
boot = np.array([
    diffs[rng.integers(0, len(diffs), len(diffs))].mean()
    for _ in range(10_000)
])
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Mean improvement: {diffs.mean():+.4f}")
print(f"95% Confidence Interval: [{lo:+.4f}, {hi:+.4f}]")

Mean improvement: +0.0282
95% Confidence Interval: [+0.0141, +0.0425]
